# 01 建库 + 知识库入库 + 检索验证

本 notebook 完成：数据库建表、RAG 入库流水线、GraphRAG 检索验证。

| Cell | 内容 |
|---|---|
| 1 | 连接 PostgreSQL，create_all() 建表 + HNSW 索引 |
| 2 | 基本 CRUD 验证 |
| 3 | md 切分测试 |
| 4 | 向量化测试 |
| 5 | 入库示例知识库 |
| 6 | 查看知识图谱 |
| 7 | 向量检索测试 |
| 8 | GraphRAG 检索测试 |
| 9 | 边类型优先级测试 |
| 10 | 空检索兜底测试 |

In [ ]:
# Cell 1: 连接数据库 + 建表 + HNSW 索引
from teaching_agent.db import init_db, async_engine
from sqlalchemy import text

await init_db()
print("✓ 表创建成功")

# 创建 HNSW 向量索引（cosine 距离）
async with async_engine.begin() as conn:
    await conn.execute(text(
        "CREATE INDEX IF NOT EXISTS idx_chunk_embedding_hnsw "
        "ON chunk USING hnsw (embedding vector_cosine_ops)"
    ))
print("✓ HNSW 索引创建成功")

In [ ]:
# Cell 2: 基本 CRUD 验证
from teaching_agent.models import Session, StudentProfile
from teaching_agent.db import async_session_factory
from datetime import datetime, timezone

async with async_session_factory() as session:
    # 清理可能残留的测试数据
    existing = await session.get(Session, "test-session-1")
    if existing:
        await session.delete(existing)
    existing_p = await session.get(StudentProfile, "profile-1")
    if existing_p:
        await session.delete(existing_p)
    await session.commit()

    # 创建 demo session
    s = Session(
        id="test-session-1", user_id="demo-student", type="free",
        status="active", started_at=datetime.now(timezone.utc),
    )
    session.add(s)

    # 创建 demo 画像
    p = StudentProfile(
        id="profile-1", user_id="demo-student",
        summary="新学生", weak_points=[], learning_style="未知",
    )
    session.add(p)
    await session.commit()
    print("✓ CRUD 验证成功")

    # 查询验证
    result = await session.get(Session, "test-session-1")
    print(f"✓ 查询: type={result.type}, status={result.status}")

    # 清理测试数据
    await session.delete(result)
    await session.delete(await session.get(StudentProfile, "profile-1"))
    await session.commit()
    print("✓ 清理完成")

In [ ]:
# Cell 3: md 切分测试
from teaching_agent.rag.chunker import chunk_markdown
from teaching_agent.knowledge.sample_data import COURSE_DOCUMENTS

doc = COURSE_DOCUMENTS[0]
chunks = chunk_markdown(doc["content"])
print(f"文档: {doc['filename']}")
print(f"切分出 {len(chunks)} 个 chunk:")
for c in chunks[:10]:
    print(f"  [{c.chunk_index}] {c.heading_path} ({len(c.content)} 字符)")
    print(f"      {c.content[:80]}...")
if len(chunks) > 10:
    print(f"  ... 还有 {len(chunks) - 10} 个 chunk")

In [ ]:
# Cell 4: 向量化测试
from teaching_agent.agents.llm import get_embedder

embedder = get_embedder()
texts = [c.content for c in chunks[:3]]
vectors = await embedder.aembed_documents(texts)
print(f"向量化 {len(vectors)} 个 chunk")
print(f"维度: {len(vectors[0])}")
print(f"前5维: {vectors[0][:5]}")

In [2]:
# Cell 5: 入库示例知识库（切分 -> 向量化 -> 写入 pgvector -> LLM 知识图谱分析）
# 注意：此 cell 会对每个 chunk 调用 LLM 进行知识图谱分析，4 篇文档约 200+ chunks，耗时较长
from teaching_agent.rag.pipeline import ingest_document
from teaching_agent.knowledge.sample_data import COURSE_DOCUMENTS
from teaching_agent.db import async_engine
from sqlalchemy import text

# 清空旧数据（重新入库时用）
async with async_engine.begin() as conn:
    await conn.execute(text("DELETE FROM chunkknowledgepoint"))
    await conn.execute(text("DELETE FROM knowledgeedge"))
    await conn.execute(text("DELETE FROM knowledgepoint"))
    await conn.execute(text("DELETE FROM chunk"))
    await conn.execute(text("DELETE FROM kbdocument"))

# 入库 4 篇课程文档
for doc in COURSE_DOCUMENTS[:1]:
    await ingest_document(
        filename=doc["filename"],
        content=doc["content"],
        scope=doc["scope"],
    )

✓ 入库完成: 物理层.md (48 chunks)


In [2]:
# Cell 6: 查看知识图谱全貌（节点 + 边）
from teaching_agent.db import async_session_factory
from teaching_agent.rag.graph import get_graph_view

async with async_session_factory() as session:
    graph = await get_graph_view(session)

print(f"知识点（{len(graph['nodes'])} 个）:")
for node in graph["nodes"]:
    print(f"  - {node['name']}: {node['description']}")

print(f"\n边（{len(graph['edges'])} 条）:")
for edge in graph["edges"]:
    print(f"  {edge['source']} --{edge['relation_type']}--> {edge['target']}")

知识点（142 个）:
  - 10G EPON: 下一代PON标准，速率可达10 Gbit/s
  - 10G GPON: 下一代PON标准，速率可达10 Gbit/s
  - 16QAM: QAM的一种具体实现，每个码元携带4比特信息。
  - 256QAM: QAM的一种具体实现，每个码元携带8比特信息。
  - 50Ω 同轴电缆: 特性阻抗为50欧姆的同轴电缆，主要用于基带传输（数字信号），如早期局域网标准10Base-2和10Base-5。
  - 64QAM: QAM的一种具体实现，每个码元携带6比特信息。
  - 75Ω 同轴电缆: 特性阻抗为75欧姆的同轴电缆，主要用于宽带传输（模拟信号），如有线电视网CATV和HFC接入。
  - ADSL: 非对称数字用户线路，利用频分复用技术在电话线上分离语音和数据信号。
  - Cable Modem: 线缆调制解调器，HFC网络中用于用户端信号调制解调的设备。
  - CATV: 有线电视系统，早期广泛采用频分复用（FDM）技术传输多路视频和音频信号。
  - CDMA: 码分多址，利用伪随机码进行扩频通信的技术。
  - CDMA移动通信: 码分多址移动通信系统，利用码分复用（CDM）原理实现多用户同时同频通信。
  - CSMA/CD: 载波监听多路访问/冲突检测协议，集线器连接的以太网采用的介质访问控制方法。
  - E1/T1: 一种数字传输标准，采用时分复用（TDM）技术将多个低速信道复用到高速信道中。
  - EPON: 基于以太网的PON标准，上下行速率均为1.25 Gbit/s
  - FTTB: 光纤到大楼，在大楼内部使用局域网技术连接用户
  - FTTC: 光纤到路边，从路边到用户使用铜线接入
  - FTTH: 光纤到户，将光纤直接拉到用户家中的宽带接入方式
  - GPON: 千兆PON标准，下行2.5 Gbit/s，上行1.25 Gbit/s
  - HFC: 混合光纤同轴网，一种宽带接入技术，结合光纤和同轴电缆传输介质。
  - ISDN: 综合业务数字网，基于时分复用（TDM）技术提供端到端的数字连接。
  - NRZ编码: 不归零编码，一种最简单的数字数据编码方式，高/低电平分别代表1和0，电平在码元周期内保持不变。
  - ODN: 光分配网络，由光纤和分光器组成，负责光

In [5]:
# Cell 7: 向量检索测试 -- 不同 query 的 top-5
# 检索逻辑：query 向量化 -> 与所有 chunk 的 embedding 算 cosine 距离 -> 距离 < 0.5 的返回
# dist=0 表示完全相同，dist=1 表示完全相反，相似度 = 1 - dist
from teaching_agent.rag.retriever import graphrag_search

queries = ["物理层的作用", "曼彻斯特编码", "CSMA/CD工作原理", "CRC循环冗余校验", "交换机自学习"]

async with async_session_factory() as session:
    for query in queries:
        chunks = await graphrag_search(session, query, "free")
        print(f"\n{'='*60}")
        print(f"Query: {query}")
        print(f"命中 {len(chunks)} 个 chunk (相似度 = 1 - dist):")
        for c in chunks:
            sim = 1 - c['dist']
            print(f"  [{c['source']}] 相似度={sim:.1%} | {c['heading_path']}")
            print(f"    {c['content'][:80]}...")


Query: 物理层的作用
命中 10 个 chunk (相似度 = 1 - dist):
  [vector] 相似度=78.9% | 物理层 > 一、物理层概述 > 1.1 物理层的基本概念
    物理层（Physical Layer）是 OSI 参考模型的最底层，也是 TCP/IP 模型中的最低层。物理层的主要任务是在物理介质上透明地传输原始的比特流。所...
  [vector] 相似度=72.9% | 物理层 > 一、物理层概述 > 1.3 物理层与数据链路层的关系
    物理层为数据链路层提供服务，将数据链路层交下来的帧转换为比特流，再通过物理介质传输。接收端的物理层则将接收到的比特流传递给数据链路层进行帧的识别和处理。两者之间...
  [vector] 相似度=68.9% | 物理层 > 一、物理层概述 > 1.2 物理层的主要任务
    物理层定义了物理连接的以下四个方面的接口特性：

1. **机械特性**：指明接口所用接线器的形状和尺寸、引脚数目和排列、固定和锁定装置等。例如，RJ-45 连...
  [vector] 相似度=57.9% | 物理层 > 六、物理层设备 > 6.1 中继器（Repeater）
    中继器是物理层设备，工作在 OSI 模型的第一层。中继器的主要功能是对信号进行再生和放大，以延长网络的传输距离。

由于信号在传输介质中传输时会逐渐衰减和失真，...
  [vector] 相似度=53.6% | 物理层 > 八、练习题 > 选择题
    **1. 物理层的四个接口特性中，指明接口所用接线器的形状和尺寸、引脚数目和排列的是（ ）。**

A. 电气特性  
B. 机械特性  
C. 功能特性  
...
  [graph:contrast] 相似度=50.6% | 物理层 > 六、物理层设备 > 6.2 集线器（Hub）
    集线器是中继器的一种多端口形式，也工作在物理层。集线器有多个端口，可以连接多台设备。

集线器的工作原理：
- 当集线器的某个端口接收到信号时，它会将信号**广...
  [graph:part_of] 相似度=50.6% | 物理层 > 六、物理层设备 > 6.2 集线器（Hub）
    集线器是中继器的一种多端口形式，也工作在物理层。集线器有多个端口，可以连接多台设备。



In [6]:
# Cell 8: GraphRAG 检索测试 -- 验证图遍历扩展效果
async with async_session_factory() as session:
    chunks = await graphrag_search(session, "数据链路层封装成帧", "free")
    print("GraphRAG 检索结果:")
    for c in chunks:
        print(f"  [{c['source']}] dist={c['dist']:.4f} | {c['heading_path']}")
        print(f"    {c['content'][:60]}...")

GraphRAG 检索结果:
  [vector] dist=0.2809 | 物理层 > 一、物理层概述 > 1.3 物理层与数据链路层的关系
    物理层为数据链路层提供服务，将数据链路层交下来的帧转换为比特流，再通过物理介质传输。接收端的物理层则将接收到的比特流传递...
  [vector] dist=0.4351 | 物理层 > 八、练习题 > 选择题
    **1. 物理层的四个接口特性中，指明接口所用接线器的形状和尺寸、引脚数目和排列的是（ ）。**

A. 电气特性  
...
  [vector] dist=0.4476 | 物理层 > 参考答案 > 选择题答案
    **1. B。** 机械特性指明接口所用接线器的形状和尺寸、引脚数目和排列、固定和锁定装置等。电气特性指明电压范围；功能...
  [vector] dist=0.4536 | 物理层 > 四、编码与调制技术 > 4.2 数字数据的数字编码 > 4.2.3 差分曼彻斯特编码（Differential Manchester Encoding）
    差分曼彻斯特编码是曼彻斯特编码的变体。每个比特周期的中间仍然有一次电平跳变（用于同步），但比特值由周期开始时是否有电平跳...
  [vector] dist=0.4701 | 物理层 > 一、物理层概述 > 1.1 物理层的基本概念
    物理层（Physical Layer）是 OSI 参考模型的最底层，也是 TCP/IP 模型中的最低层。物理层的主要任务...
  [graph:part_of] dist=0.4706 | 物理层 > 一、物理层概述 > 1.2 物理层的主要任务
    物理层定义了物理连接的以下四个方面的接口特性：

1. **机械特性**：指明接口所用接线器的形状和尺寸、引脚数目和排列...
  [graph:part_of] dist=0.4963 | 物理层 > 二、数据通信基础 > 2.1 数据通信系统的模型
    一个数据通信系统可划分为三大部分：源系统（发送端）、传输系统（传输网络）和目的系统（接收端）。

- **源系统**包括...
  [graph:contrast] dist=0.4970 | 物理层 > 四、编码与调制技术 > 4.2 数字数据的数字编码 > 三种编码方式的对比
 

In [11]:
# Cell 9: 边类型优先级测试 -- 查看图遍历是否正确扩展了 prerequisite/contrast/related 的 chunk
async with async_session_factory() as session:
    chunks = await graphrag_search(session, "CSMA/CD协议", "free")
    graph_chunks = [c for c in chunks if c["source"].startswith("graph")]
    print(f"图扩展结果 ({len(graph_chunks)} 个):")
    for c in graph_chunks:
        print(f"  {c['source']} | {c['heading_path']}")
    if not graph_chunks:
        print("  （本次检索无图扩展结果，可能知识点间无边或已被向量结果覆盖）")

图扩展结果 (5 个):
  graph:contrast | 物理层 > 四、编码与调制技术 > 4.2 数字数据的数字编码 > 4.2.3 差分曼彻斯特编码（Differential Manchester Encoding）
  graph:prerequisite | 物理层 > 四、编码与调制技术 > 4.2 数字数据的数字编码 > 4.2.3 差分曼彻斯特编码（Differential Manchester Encoding）
  graph:related | 物理层 > 四、编码与调制技术 > 4.2 数字数据的数字编码 > 4.2.3 差分曼彻斯特编码（Differential Manchester Encoding）
  graph:contrast | 物理层 > 八、练习题 > 简答题
  graph:part_of | 物理层 > 八、练习题 > 简答题


In [13]:
# Cell 10: 空检索兜底测试
async with async_session_factory() as session:
    chunks = await graphrag_search(session, "量子力学原理", "free")
    print(chunks)

[]
